# A5 – Collaborative Filtering Recommender Systems
**Student:** Joud Taher  
**Dataset:** Amazon Beauty Reviews (explicit 1–5 star ratings)  

## Objective
Build, evaluate, and compare collaborative filtering (CF) recommender systems on the Amazon Beauty dataset. We explore user-based CF, item-based CF, and model-based CF (matrix factorisation via SVD), benchmarked against non-personalised baselines. The bonus section provides an in-depth interpretation of SVD latent factors.

Collaborative filtering recommends items to users based purely on the collective pattern of ratings—no item content or user demographics are needed. This makes it well-suited for large, sparse, explicit-feedback datasets such as Amazon product reviews.

---
## Section 1 – Setup & Imports

In [ ]:
!pip install scikit-surprise kagglehub -q

In [ ]:
import os
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

from surprise import Dataset, Reader, SVD, KNNBasic, KNNWithMeans, NormalPredictor, BaselineOnly
from surprise import accuracy
from surprise.model_selection import train_test_split, cross_validate, GridSearchCV

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 110
print("All imports OK")

---
## Section 2 – Dataset Loading & Core Statistics

In [ ]:
# ── Setup ──────────────────────────────────────────────────────────────────
import os
import kagglehub

# Download Amazon Beauty ratings via kagglehub
dataset_path = kagglehub.dataset_download("skillsmuggler/amazon-ratings")
file_path = os.path.join(dataset_path, "ratings_Beauty.csv")

amazon_df = pd.read_csv(file_path)
amazon_df = amazon_df.rename(columns={
    "UserId":    "user_id",
    "ProductId": "item_id",
    "Rating":    "rating",
    "Timestamp": "timestamp"
})
amazon_df = amazon_df[["user_id", "item_id", "rating", "timestamp"]].dropna().copy()
amazon_df["rating"] = pd.to_numeric(amazon_df["rating"], errors="coerce")
amazon_df = amazon_df.dropna(subset=["rating"]).copy()

In [ ]:
print(amazon_df.head())
print("Shape:", amazon_df.shape)
print(amazon_df.dtypes)

In [ ]:
user_activity  = amazon_df.groupby("user_id").size()
item_popularity = amazon_df.groupby("item_id").size()
n_users   = amazon_df["user_id"].nunique()
n_items   = amazon_df["item_id"].nunique()
n_ratings = len(amazon_df)
density   = n_ratings / (n_users * n_items)
sparsity  = 1.0 - density

print(f"Total ratings : {n_ratings:,}")
print(f"Unique users  : {n_users:,}")
print(f"Unique items  : {n_items:,}")
print(f"Rating range  : {amazon_df['rating'].min()} \u2013 {amazon_df['rating'].max()}")
print(f"Mean rating   : {amazon_df['rating'].mean():.2f}")
print(f"Sparsity      : {sparsity:.4%}")

### Interpretation
The dataset contains roughly 2 million explicit ratings from hundreds of thousands of users across tens of thousands of products. The extreme sparsity (>99.9%) is typical of real-world e-commerce data—most user–item pairs are unobserved. The mean rating above 4 indicates strong **positivity bias**: users are much more likely to rate items they liked. This has implications for evaluation (accuracy metrics alone can be misleading) and for model design (the model must generalise from a very small fraction of possible ratings).

---
## Section 3 – Exploratory Data Analysis (EDA)

### 3.1 Rating Distribution

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
rating_counts = amazon_df["rating"].value_counts().sort_index()
ax.bar(rating_counts.index.astype(int), rating_counts.values, color="steelblue", edgecolor="white")
ax.set_xlabel("Rating", fontsize=12)
ax.set_ylabel("Count", fontsize=12)
ax.set_title("Rating Distribution", fontsize=14)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x/1e6:.1f}M" if x >= 1e6 else f"{int(x):,}"))
plt.tight_layout()
plt.show()

### Reflection – Rating Distribution
The rating distribution is heavily **right-skewed**: 5-star ratings dominate, followed by 4-star, with very few 1- or 2-star ratings. This **positivity bias** means the rating scale is compressed in practice—most items receive high scores. For evaluation, RMSE/MAE will be influenced by how well the model predicts these high ratings. For recommendations, a model that always predicts high ratings might score deceptively well on accuracy metrics, so diversity and novelty metrics would complement RMSE/MAE.

### 3.2 User Activity Distribution

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(user_activity, bins=60, color="coral", edgecolor="white", log=True)
ax.set_xlabel("Ratings per User", fontsize=12)
ax.set_ylabel("Number of Users (log scale)", fontsize=12)
ax.set_title("User Activity Distribution", fontsize=14)
plt.tight_layout()
plt.show()

print("User activity quantiles:")
for q in [0.25, 0.50, 0.75, 0.90, 0.95]:
    print(f"  {int(q*100)}th pct: {user_activity.quantile(q):.0f} ratings")

### Reflection – User Activity
User activity follows a **power-law / long-tail distribution**: most users have rated only a handful of items, while a small number of highly active users dominate the rating pool. The median user has very few ratings, making it difficult to personalise recommendations for them (cold-start problem). Collaborative filtering degrades for these sparse users; additional heuristics or content-based fallbacks would be needed in production.

### 3.3 Item Popularity Distribution

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(item_popularity, bins=60, color="mediumseagreen", edgecolor="white", log=True)
ax.set_xlabel("Ratings per Item", fontsize=12)
ax.set_ylabel("Number of Items (log scale)", fontsize=12)
ax.set_title("Item Popularity Distribution", fontsize=14)
plt.tight_layout()
plt.show()

print("Item popularity quantiles:")
for q in [0.25, 0.50, 0.75, 0.90, 0.95]:
    print(f"  {int(q*100)}th pct: {item_popularity.quantile(q):.0f} ratings")

### Reflection – Item Popularity
Like user activity, item popularity also has a **long-tail**: a small fraction of items (blockbuster products) attract a disproportionate number of ratings, while the vast majority are rated by very few users. This **popularity bias** can cause CF models to over-recommend popular items, reducing diversity and making it harder to surface niche products that might actually match a user's preferences.

### 3.4 Sparsity Visualisation

In [ ]:
fig, ax = plt.subplots(figsize=(5, 3))
ax.bar(["Observed", "Missing"], [density * 100, sparsity * 100],
       color=["steelblue", "lightgrey"], edgecolor="white")
ax.set_ylabel("%", fontsize=12)
ax.set_title(f"User–Item Matrix Fill Rate  (sparsity = {sparsity:.4%})", fontsize=12)
for i, v in enumerate([density * 100, sparsity * 100]):
    ax.text(i, v + 0.1, f"{v:.3f}%", ha="center", fontsize=11)
plt.tight_layout()
plt.show()

### Reflection – Sparsity
The user–item matrix is over 99.9% empty. For memory-based CF (user-based and item-based), this means finding neighbours with enough overlap is difficult—cosine or Pearson similarity computed on sparse vectors is noisy. For model-based CF (SVD), the goal is precisely to fill in these missing entries by learning low-rank latent representations, making it more robust to sparsity.

### 3.5 Key EDA Observations

1. **Positivity bias**: Ratings are heavily skewed toward 5 stars; RMSE/MAE metrics may understate the challenge of predicting low ratings.
2. **User activity skew**: Most users rated very few items, creating a severe cold-start problem for memory-based CF.
3. **Item popularity skew**: A small number of bestseller products account for the majority of ratings; CF risks over-recommending popular items.
4. **Extreme sparsity (>99.9%)**: Neighbourhood methods rely on overlapping ratings; with so few overlaps, similarity estimates are noisy. Matrix factorisation (SVD) is better suited to this regime.
5. **Explicit feedback**: Ratings are on a 1–5 scale, enabling regression-based evaluation (RMSE, MAE) rather than ranking-only metrics.

---
## Section 4 – Preprocessing

In [ ]:
# Before filtering
print(f"Before filtering: {len(amazon_df):,} ratings")

user_counts = amazon_df.groupby("user_id")["item_id"].count()
item_counts = amazon_df.groupby("item_id")["user_id"].count()

valid_users = user_counts[user_counts >= 5].index
valid_items = item_counts[item_counts >= 5].index

amazon_df_filtered = amazon_df[
    amazon_df["user_id"].isin(valid_users) &
    amazon_df["item_id"].isin(valid_items)
].copy()

print(f"After filtering : {len(amazon_df_filtered):,} ratings")
print(f"Retained        : {len(amazon_df_filtered)/len(amazon_df):.1%} of original data")

### Reflection – Preprocessing
The threshold of **5 ratings** per user and per item was chosen as a pragmatic balance: it removes the most extreme cold-start cases (users/items with only 1–4 ratings, where similarity estimates are unreliable) while retaining the majority of the data. A higher threshold (e.g., 20) would produce a denser, more reliable matrix but discard many users and items; a lower threshold preserves more data at the cost of noisy neighbours. The 5-rating cut-off is standard in CF literature for Amazon review datasets.

---
## Section 5 – Evaluation Setup

### Evaluation Strategy
Since we have **explicit feedback** (1–5 star ratings), we use regression error metrics:
- **RMSE** (Root Mean Squared Error): penalises large errors more heavily; lower is better.
- **MAE** (Mean Absolute Error): average absolute deviation; lower is better.

We split the filtered data **80% train / 20% test** (with `random_state=42`) for held-out evaluation, and use **5-fold cross-validation** for more robust, variance-reduced estimates. All models are compared against non-personalised baselines to confirm that personalisation adds value.

In [ ]:
from surprise import Dataset, Reader
from surprise.model_selection import train_test_split, cross_validate

reader = Reader(rating_scale=(1, 5))
surprise_data = Dataset.load_from_df(
    amazon_df_filtered[["user_id", "item_id", "rating"]],
    reader
)
trainset, testset = train_test_split(surprise_data, test_size=0.2, random_state=42)
print(f"Trainset size: {trainset.n_ratings:,} ratings")
print(f"Testset size : {len(testset):,} ratings")

---
## Section 6 – Baseline Models (Non-Personalised)

### 6.1 Random Recommender (`NormalPredictor`)

In [ ]:
results = {}

# Random baseline
rng_model = NormalPredictor()
rng_model.fit(trainset)
rng_preds = rng_model.test(testset)
results["Random"] = {
    "RMSE": accuracy.rmse(rng_preds, verbose=False),
    "MAE":  accuracy.mae(rng_preds,  verbose=False),
}
print(f"Random  RMSE={results['Random']['RMSE']:.4f}  MAE={results['Random']['MAE']:.4f}")

### 6.2 Global Mean Baseline

In [ ]:
global_mean = trainset.global_mean
gm_preds = [(uid, iid, r_ui, global_mean, {}) for (uid, iid, r_ui) in testset]
from surprise import Prediction
gm_predictions = [Prediction(uid, iid, r_ui, est, details)
                  for uid, iid, r_ui, est, details in gm_preds]
results["GlobalMean"] = {
    "RMSE": accuracy.rmse(gm_predictions, verbose=False),
    "MAE":  accuracy.mae(gm_predictions,  verbose=False),
}
print(f"GlobalMean  RMSE={results['GlobalMean']['RMSE']:.4f}  MAE={results['GlobalMean']['MAE']:.4f}")

### 6.3 Bias-Only Baseline (`BaselineOnly`)

In [ ]:
bl_model = BaselineOnly(verbose=False)
bl_model.fit(trainset)
bl_preds = bl_model.test(testset)
results["BiasOnly"] = {
    "RMSE": accuracy.rmse(bl_preds, verbose=False),
    "MAE":  accuracy.mae(bl_preds,  verbose=False),
}
print(f"BiasOnly  RMSE={results['BiasOnly']['RMSE']:.4f}  MAE={results['BiasOnly']['MAE']:.4f}")

---
## Section 7 – Memory-Based Collaborative Filtering

### 7.1 User-Based CF (`KNNWithMeans`)

In [ ]:
user_cf = KNNWithMeans(k=40, sim_options={"name": "pearson", "user_based": True}, verbose=False)
user_cf.fit(trainset)
user_preds = user_cf.test(testset)
results["UserCF"] = {
    "RMSE": accuracy.rmse(user_preds, verbose=False),
    "MAE":  accuracy.mae(user_preds,  verbose=False),
}
print(f"UserCF  RMSE={results['UserCF']['RMSE']:.4f}  MAE={results['UserCF']['MAE']:.4f}")

### 7.2 Item-Based CF (`KNNWithMeans`)

In [ ]:
item_cf = KNNWithMeans(k=40, sim_options={"name": "pearson", "user_based": False}, verbose=False)
item_cf.fit(trainset)
item_preds = item_cf.test(testset)
results["ItemCF"] = {
    "RMSE": accuracy.rmse(item_preds, verbose=False),
    "MAE":  accuracy.mae(item_preds,  verbose=False),
}
print(f"ItemCF  RMSE={results['ItemCF']['RMSE']:.4f}  MAE={results['ItemCF']['MAE']:.4f}")

---
## Section 8 – Model-Based CF (SVD)

In [ ]:
svd_model = SVD(n_factors=100, n_epochs=20, lr_all=0.005, reg_all=0.02, random_state=42)
svd_model.fit(trainset)
svd_preds = svd_model.test(testset)
results["SVD"] = {
    "RMSE": accuracy.rmse(svd_preds, verbose=False),
    "MAE":  accuracy.mae(svd_preds,  verbose=False),
}
print(f"SVD  RMSE={results['SVD']['RMSE']:.4f}  MAE={results['SVD']['MAE']:.4f}")

---
## Section 9 – Cross-Validation & Model Comparison

In [ ]:
cv_models = {
    "Random":   NormalPredictor(),
    "BiasOnly": BaselineOnly(verbose=False),
    "UserCF":   KNNWithMeans(k=40, sim_options={"name": "pearson", "user_based": True},  verbose=False),
    "ItemCF":   KNNWithMeans(k=40, sim_options={"name": "pearson", "user_based": False}, verbose=False),
    "SVD":      SVD(n_factors=100, n_epochs=20, lr_all=0.005, reg_all=0.02, random_state=42),
}

cv_results = {}
for name, model in cv_models.items():
    cv = cross_validate(model, surprise_data, measures=["RMSE", "MAE"], cv=5, verbose=False)
    cv_results[name] = {
        "RMSE_mean": cv["test_rmse"].mean(),
        "RMSE_std":  cv["test_rmse"].std(),
        "MAE_mean":  cv["test_mae"].mean(),
        "MAE_std":   cv["test_mae"].std(),
    }
    print(f"{name:12s}  CV RMSE={cv_results[name]['RMSE_mean']:.4f}±{cv_results[name]['RMSE_std']:.4f}  "
          f"MAE={cv_results[name]['MAE_mean']:.4f}±{cv_results[name]['MAE_std']:.4f}")

In [ ]:
# Comparison bar chart
cv_df = pd.DataFrame(cv_results).T
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, metric in zip(axes, ["RMSE", "MAE"]):
    means = cv_df[f"{metric}_mean"]
    stds  = cv_df[f"{metric}_std"]
    bars = ax.bar(means.index, means.values, yerr=stds.values, capsize=5,
                  color=sns.color_palette("muted", len(means)))
    ax.set_title(f"5-Fold CV {metric}", fontsize=13)
    ax.set_ylabel(metric)
    ax.set_ylim(0, means.max() * 1.25)
    ax.tick_params(axis="x", rotation=30)
    for bar, val in zip(bars, means.values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                f"{val:.3f}", ha="center", va="bottom", fontsize=9)
plt.suptitle("Model Comparison – Cross-Validated RMSE & MAE", fontsize=14)
plt.tight_layout()
plt.show()

### Reflection – Model Comparison
SVD consistently achieves the lowest RMSE and MAE across folds, confirming that matrix factorisation is better suited to this sparse dataset than neighbourhood methods. User-based and item-based CF outperform random but are close to the bias-only baseline, illustrating the difficulty of finding reliable neighbours in a very sparse matrix. The error bars from 5-fold CV indicate that the ranking of models is stable across different train/test splits.

---
## Section 10 – Hyperparameter Analysis

### 10.1 SVD – Number of Latent Factors

In [ ]:
factor_values = [2, 10, 50, 100, 200]
factor_rmse   = []
for k in factor_values:
    m = SVD(n_factors=k, n_epochs=20, lr_all=0.005, reg_all=0.02, random_state=42)
    m.fit(trainset)
    p = m.test(testset)
    factor_rmse.append(accuracy.rmse(p, verbose=False))
    print(f"  n_factors={k:4d}  RMSE={factor_rmse[-1]:.4f}")

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(factor_values, factor_rmse, marker="o", color="steelblue")
ax.set_xlabel("Number of Latent Factors", fontsize=12)
ax.set_ylabel("RMSE", fontsize=12)
ax.set_title("SVD: RMSE vs. Number of Latent Factors", fontsize=13)
ax.set_xscale("log")
plt.tight_layout()
plt.show()

### Reflection – Number of Latent Factors
Too few factors (e.g., 2) under-fit because the model cannot capture enough rating patterns. Performance improves as factors increase up to a point, then plateaus or slightly worsens due to overfitting. The optimal range for this dataset is roughly 50–100 factors: beyond that, regularisation struggles to prevent over-fitting on infrequent users/items.

### 10.2 SVD – Regularisation Strength

In [ ]:
reg_values = [0.001, 0.01, 0.02, 0.1, 0.5]
reg_rmse   = []
for r in reg_values:
    m = SVD(n_factors=100, n_epochs=20, lr_all=0.005, reg_all=r, random_state=42)
    m.fit(trainset)
    p = m.test(testset)
    reg_rmse.append(accuracy.rmse(p, verbose=False))
    print(f"  reg_all={r:.3f}  RMSE={reg_rmse[-1]:.4f}")

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(reg_values, reg_rmse, marker="s", color="coral")
ax.set_xlabel("Regularisation (reg_all)", fontsize=12)
ax.set_ylabel("RMSE", fontsize=12)
ax.set_title("SVD: RMSE vs. Regularisation Strength", fontsize=13)
ax.set_xscale("log")
plt.tight_layout()
plt.show()

### Reflection – Regularisation
Very low regularisation (0.001) over-fits because the model memorises training ratings with minimal penalty on factor magnitudes. Very high regularisation (0.5) under-fits because the factors are forced to stay near zero regardless of the data. The sweet spot around 0.01–0.02 balances fit and generalisation for this dataset size.

### 10.3 KNN – Number of Neighbours

In [ ]:
k_values   = [5, 10, 20, 40, 80]
ubcf_rmse  = []
ibcf_rmse  = []
for k in k_values:
    ub = KNNWithMeans(k=k, sim_options={"name": "pearson", "user_based": True},  verbose=False)
    ib = KNNWithMeans(k=k, sim_options={"name": "pearson", "user_based": False}, verbose=False)
    ub.fit(trainset); ib.fit(trainset)
    ubcf_rmse.append(accuracy.rmse(ub.test(testset), verbose=False))
    ibcf_rmse.append(accuracy.rmse(ib.test(testset), verbose=False))
    print(f"  k={k:3d}  UserCF RMSE={ubcf_rmse[-1]:.4f}  ItemCF RMSE={ibcf_rmse[-1]:.4f}")

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(k_values, ubcf_rmse, marker="o", label="User-Based CF")
ax.plot(k_values, ibcf_rmse, marker="s", label="Item-Based CF")
ax.set_xlabel("Number of Neighbours (k)", fontsize=12)
ax.set_ylabel("RMSE", fontsize=12)
ax.set_title("KNN CF: RMSE vs. Number of Neighbours", fontsize=13)
ax.legend()
plt.tight_layout()
plt.show()

### Reflection – Number of Neighbours
For user-based CF, very few neighbours (k=5) gives noisy predictions because similarity estimates are unreliable in sparse data; increasing k smooths predictions up to a point, after which including distant (less similar) neighbours introduces noise again. Item-based CF is generally less sensitive to k because item similarity vectors are denser (items have more ratings per item on average than users). Both methods perform worse than SVD overall, confirming the limitations of memory-based approaches on this scale.

---
## Section 11 – Bonus: Latent Factor Analysis (SVD)

### 11.1 PCA Projection of Item Latent Factors

In [ ]:
from sklearn.decomposition import PCA

# Retrain SVD on the full trainset for factor analysis
svd_full = SVD(n_factors=100, n_epochs=20, lr_all=0.005, reg_all=0.02, random_state=42)
svd_full.fit(trainset)

# Item factor matrix  (n_items × n_factors)
Qi = svd_full.qi  # shape: (n_items_in_trainset, n_factors)

# PCA down to 2 dimensions for visualisation
pca = PCA(n_components=2, random_state=42)
Qi_2d = pca.fit_transform(Qi)

fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(Qi_2d[:, 0], Qi_2d[:, 1], s=4, alpha=0.3, color="steelblue")
ax.set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]:.1%} var)", fontsize=12)
ax.set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]:.1%} var)", fontsize=12)
ax.set_title("PCA Projection of Item Latent Factors (SVD, 100 factors)", fontsize=13)
plt.tight_layout()
plt.show()

print(f"Total variance explained by first 2 PCs: "
      f"{pca.explained_variance_ratio_.sum():.1%}")

### 11.2 User Latent Factor Distribution

In [ ]:
Pu = svd_full.pu  # shape: (n_users_in_trainset, n_factors)
Pu_2d = pca.transform(Pu[:, :Qi.shape[1]])  # reuse same PCA projection

fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(Pu_2d[:, 0], Pu_2d[:, 1], s=4, alpha=0.2, color="coral")
ax.set_xlabel(f"PC1", fontsize=12)
ax.set_ylabel(f"PC2", fontsize=12)
ax.set_title("PCA Projection of User Latent Factors (SVD, 100 factors)", fontsize=13)
plt.tight_layout()
plt.show()

### 11.3 Sample Recommendations for the Most Active User

In [ ]:
most_active_user = amazon_df_filtered.groupby("user_id")["item_id"].count().idxmax()
rated_items   = set(amazon_df_filtered[amazon_df_filtered["user_id"] == most_active_user]["item_id"])
all_items     = set(amazon_df_filtered["item_id"].unique())
unrated_items = all_items - rated_items

# Predict ratings for unrated items and show top-10
predictions = [
    (item, svd_full.predict(most_active_user, item).est)
    for item in unrated_items
]
top10 = sorted(predictions, key=lambda x: x[1], reverse=True)[:10]

print(f"Most active user : {most_active_user}")
print(f"Items rated      : {len(rated_items)}")
print(f"Unrated items    : {len(unrated_items)}")
print("\nTop-10 recommended items (predicted rating):")
for rank, (item_id, est) in enumerate(top10, 1):
    print(f"  {rank:2d}. {item_id}  (est. rating: {est:.2f})")

### 11.4 In-Depth Interpretation of Latent Factors

SVD decomposes the user–item rating matrix **R ≈ U Σ Vᵀ**, learning:
- **User factor matrix Pu**: each user is represented as a dense vector in a low-dimensional latent space. The dimensions capture abstract taste dimensions (e.g., preference for luxury skincare vs. budget haircare).
- **Item factor matrix Qi**: each product is represented similarly. Items close together in latent space tend to receive similar ratings from the same users.
- **Bias terms (bu, bi)**: offset global mean to account for users who rate systematically higher/lower and items that are systematically better/worse-rated.

**What the PCA scatter reveals**:  
The PCA projection compresses 100 latent dimensions to 2, explaining a small fraction of total variance. The resulting scatter shows that items (and users) form a somewhat diffuse cloud with a few outliers. This is consistent with the dataset's diversity—Amazon Beauty covers thousands of product categories (lipstick, sunscreen, shampoo, etc.), so no single dominant axis of variation is expected.

**Why latent factors are valuable**:  
Unlike memory-based CF, SVD latent factors generalise across users and items that share no direct co-rating history. Two users who have never rated any of the same items can still receive similar recommendations if their factor vectors align. This is why SVD achieves lower RMSE than KNN on this sparse dataset.

**Limitations**:  
The latent dimensions are not interpretable by inspection—they are mathematical constructs optimised for rating prediction, not human-readable categories. Additionally, the static factorisation cannot capture temporal dynamics (e.g., a product trending due to a viral review) without retraining.